# FloodLens — Phase 3: Robustness Testing
### Synthetic degradation (rain/fog/low-light/glare), classical enhancement, results table

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

This is self-contained — rebuilds the exact same data pipeline and val split as
Phase 2 (same seed, same subset size) so the "clean" IoU here should closely match
your Phase 2 best_val_iou. Loads your trained checkpoint from Drive, so **run Phase 2
first** if you haven't.


## 0. Setup

In [ ]:
!pip install -q kagglehub segmentation-models-pytorch albumentations opencv-python-headless
print("Dependencies installed.")


In [ ]:
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split
import kagglehub
import segmentation_models_pytorch as smp
import albumentations as A

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Mount Drive and load the Phase 2 checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/FloodLens_checkpoints"
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
assert os.path.exists(checkpoint_path), f"No checkpoint found at {checkpoint_path} — run Phase 2 first."
print("Checkpoint found:", checkpoint_path)


## 2. Rebuild the exact same data pipeline and val split as Phase 2

In [ ]:
path = kagglehub.dataset_download("aletbm/aerial-imagery-dataset-floodnet-challenge")

def extract_id(filename):
    match = re.match(r"(\d+)", filename)
    return match.group(1) if match else None

candidate_img_dirs, candidate_mask_dirs = [], []
for root, dirs, files in os.walk(path):
    name = os.path.basename(root).lower()
    if not files:
        continue
    if any(k in name for k in ["label", "mask", "gt", "annotation"]):
        candidate_mask_dirs.append(root)
    elif any(k in name for k in ["img", "image"]):
        candidate_img_dirs.append(root)

best_pair, best_overlap = None, -1
for img_d in candidate_img_dirs:
    img_ids = set(filter(None, (extract_id(f) for f in os.listdir(img_d))))
    for mask_d in candidate_mask_dirs:
        mask_ids = set(filter(None, (extract_id(f) for f in os.listdir(mask_d))))
        overlap = len(img_ids & mask_ids)
        if overlap > best_overlap:
            best_overlap = overlap
            best_pair = (img_d, mask_d)

image_dir, mask_dir = best_pair
print("image_dir:", image_dir)
print("mask_dir:", mask_dir)


In [ ]:
IMG_SIZE = 256

class FloodDataset(Dataset):
    def __init__(self, image_dir, mask_dir, subset_size=None, img_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        image_files = sorted(os.listdir(image_dir))
        mask_files = sorted(os.listdir(mask_dir))
        mask_by_id = {}
        for m in mask_files:
            mid = extract_id(m)
            if mid is not None:
                mask_by_id[mid] = m
        self.pairs = []
        for f in image_files:
            fid = extract_id(f)
            if fid is not None and fid in mask_by_id:
                self.pairs.append((f, mask_by_id[fid]))
        if subset_size is not None:
            self.pairs = self.pairs[:subset_size]
        print(f"Dataset initialized with {len(self.pairs)} matched image/mask pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]
        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)
        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask.astype(np.int64))
        return img, mask

# MUST match Phase 2 exactly for a fair comparison against your recorded baseline
SUBSET_SIZE = 600
dataset = FloodDataset(image_dir, mask_dir, subset_size=SUBSET_SIZE, img_size=IMG_SIZE)

val_fraction = 0.2
val_size = int(len(dataset) * val_fraction)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)   # same seed as Phase 2 = same val samples
)
print(f"Val size: {len(val_dataset)}")


## 3. Set WATER_CLASS_ID
**Use the same value you confirmed in Phase 2** — don't let this drift between
notebooks, or your "clean" IoU here won't match your recorded baseline.


In [ ]:
WATER_CLASS_ID = 5   # set to whatever you confirmed in Phase 2's distribution check

def to_binary_np(mask, water_id=WATER_CLASS_ID):
    return (mask == water_id).astype(np.uint8)


## 4. Load the trained model

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']+1}, recorded val_iou={checkpoint['val_iou']:.4f}")
print("This recorded value is what 'clean' below should closely match.")


## 5. Degradation transforms
Each simulates one condition in isolation. Kept to minimal required parameters
(mostly just `p=1.0`) since Albumentations' exact parameter names for these
transforms have changed across versions — defaults are safer than guessing a
version-specific signature.


In [ ]:
rain_transform = A.Compose([A.RandomRain(p=1.0)])
fog_transform = A.Compose([A.RandomFog(p=1.0)])
lowlight_transform = A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.6, -0.4), contrast_limit=0, p=1.0)])
glare_transform = A.Compose([A.RandomSunFlare(p=1.0)])

DEGRADATIONS = {
    "rain": rain_transform,
    "fog": fog_transform,
    "low_light": lowlight_transform,
    "glare": glare_transform,
}
print("Degradation transforms ready:", list(DEGRADATIONS.keys()))


## 6. Visual check — confirm the degradations actually look like their names

In [ ]:
def load_raw_pair(ds, idx, img_size=256):
    img_name, mask_name = ds.pairs[idx]
    img = cv2.imread(os.path.join(ds.image_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    mask = cv2.imread(os.path.join(ds.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
    return img, mask

sample_idx = val_dataset.indices[0]
raw_img, _ = load_raw_pair(dataset, sample_idx)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(raw_img)
axes[0].set_title("Clean")
for i, (name, transform) in enumerate(DEGRADATIONS.items(), start=1):
    degraded = transform(image=raw_img)["image"]
    axes[i].imshow(degraded)
    axes[i].set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Classical enhancement functions
**Dark Channel Prior dehazing** (He et al.) — estimates haze thickness per pixel and
removes it, used for fog. **Single-Scale Retinex** — separates illumination from
reflectance to recover detail in dark regions, used for low-light and glare.


In [ ]:
def dark_channel(img, size=15):
    min_channel = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    return cv2.erode(min_channel, kernel)

def estimate_atmospheric_light(img, dark):
    h, w = dark.shape
    num_pixels = h * w
    num_brightest = max(int(num_pixels * 0.001), 1)
    dark_vec = dark.reshape(num_pixels)
    img_vec = img.reshape(num_pixels, 3)
    indices = np.argsort(dark_vec)[-num_brightest:]
    return img_vec[indices].mean(axis=0)

def dehaze(img, size=15, omega=0.95, t0=0.1):
    img_f = img.astype(np.float32) / 255.0
    dark = dark_channel(img_f, size)
    A = estimate_atmospheric_light(img_f, dark)
    norm_img = img_f / np.maximum(A, 1e-6)
    transmission = 1 - omega * dark_channel(norm_img, size)
    transmission = np.clip(transmission, t0, 1)
    result = np.empty_like(img_f)
    for c in range(3):
        result[:, :, c] = (img_f[:, :, c] - A[c]) / transmission + A[c]
    result = np.clip(result, 0, 1) * 255
    return result.astype(np.uint8)

def single_scale_retinex(img, sigma=30):
    img_f = img.astype(np.float32) + 1.0
    blur = cv2.GaussianBlur(img_f, (0, 0), sigma)
    retinex = np.log(img_f) - np.log(blur + 1.0)
    retinex = cv2.normalize(retinex, None, 0, 255, cv2.NORM_MINMAX)
    return retinex.astype(np.uint8)

def enhance_for_condition(img, condition):
    if condition == "fog":
        return dehaze(img)
    elif condition == "low_light":
        return single_scale_retinex(img)
    elif condition == "glare":
        return single_scale_retinex(img)
    elif condition == "rain":
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    return img

print("Enhancement functions ready.")


## 8. Visual check — enhancement should visibly reduce the degradation

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for i, (name, transform) in enumerate(DEGRADATIONS.items()):
    degraded = transform(image=raw_img)["image"]
    enhanced = enhance_for_condition(degraded, name)

    axes[0, i].imshow(raw_img)
    axes[0, i].set_title("Clean")
    axes[1, i].imshow(degraded)
    axes[1, i].set_title(f"{name} (degraded)")
    axes[2, i].imshow(enhanced)
    axes[2, i].set_title(f"{name} (enhanced)")
    for row in range(3):
        axes[row, i].axis("off")
plt.tight_layout()
plt.show()


## 9. Evaluation function
Runs the model on the val set under a given condition (clean, degraded, or
degraded+enhanced) and returns mean IoU.


In [ ]:
def evaluate_condition(model, ds, indices, degrade_fn=None, enhance_condition=None, batch_size=8):
    model.eval()
    ious = []
    imgs_batch, masks_batch = [], []

    def flush():
        if not imgs_batch:
            return
        imgs_t = torch.stack(imgs_batch).to(device)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs_t)
            probs = torch.sigmoid(logits)
        for i in range(imgs_t.shape[0]):
            pred = (probs[i, 0].cpu().numpy() > 0.5).astype(np.uint8)
            true = masks_batch[i]
            inter = np.logical_and(pred, true).sum()
            union = np.logical_or(pred, true).sum()
            iou = 1.0 if union == 0 else inter / union
            ious.append(iou)
        imgs_batch.clear()
        masks_batch.clear()

    for idx in indices:
        img, mask = load_raw_pair(ds, idx)
        if degrade_fn is not None:
            img = degrade_fn(image=img)["image"]
        if enhance_condition is not None:
            img = enhance_for_condition(img, enhance_condition)

        img_norm = img.astype(np.float32) / 255.0
        img_t = torch.from_numpy(img_norm).permute(2, 0, 1)
        imgs_batch.append(img_t)
        masks_batch.append(to_binary_np(mask))

        if len(imgs_batch) >= batch_size:
            flush()
    flush()

    return float(np.mean(ious)), ious

print("Evaluation function ready.")


## 10. Run the full benchmark
This is the core results table. For each condition: clean IoU (same for all rows,
computed once), degraded IoU, and degraded+enhanced IoU.


In [ ]:
val_indices = val_dataset.indices

print("Evaluating clean baseline...")
clean_iou, _ = evaluate_condition(model, dataset, val_indices)
print(f"Clean IoU: {clean_iou:.4f}\n")

results = []
for condition, transform in DEGRADATIONS.items():
    print(f"Evaluating condition: {condition}")
    degraded_iou, _ = evaluate_condition(model, dataset, val_indices, degrade_fn=transform)
    enhanced_iou, _ = evaluate_condition(model, dataset, val_indices, degrade_fn=transform, enhance_condition=condition)
    results.append({
        "condition": condition,
        "clean_iou": clean_iou,
        "degraded_iou": degraded_iou,
        "enhanced_iou": enhanced_iou,
        "drop_from_degradation": clean_iou - degraded_iou,
        "recovery_from_enhancement": enhanced_iou - degraded_iou,
    })
    print(f"  degraded={degraded_iou:.4f}  enhanced={enhanced_iou:.4f}  recovery={enhanced_iou - degraded_iou:+.4f}\n")

print("Benchmark complete.")


## 11. Results table and plot

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df[["condition", "clean_iou", "degraded_iou", "enhanced_iou", "drop_from_degradation", "recovery_from_enhancement"]]
print(df.to_string(index=False))

df.to_csv(os.path.join(CHECKPOINT_DIR, "phase3_results.csv"), index=False)
print(f"\nResults saved to {CHECKPOINT_DIR}/phase3_results.csv")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(df))
width = 0.25

ax.bar(x - width, df["clean_iou"], width, label="Clean")
ax.bar(x, df["degraded_iou"], width, label="Degraded")
ax.bar(x + width, df["enhanced_iou"], width, label="Degraded + Enhanced")

ax.set_xticks(x)
ax.set_xticklabels(df["condition"])
ax.set_ylabel("IoU")
ax.set_title("Segmentation robustness under degraded conditions")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, "phase3_results_chart.png"))
plt.show()

print("Chart saved to Drive — this is your headline figure for the writeup.")


## Phase 3 complete
You now have:
- Quantified IoU drop under rain, fog, low-light, and glare
- Whether classical enhancement recovers accuracy, and by how much, per condition
- A results table (CSV) and chart saved to Drive

**Be honest in your writeup about what this shows.** If enhancement barely helps for
some condition, report that — a partial or negative result with a clear explanation
is more credible than every number conveniently improving.

**Next:** Phase 4 (local Karachi data, optional) or move to Phase 5 (WebXR/AR layer)
if you want to build the visualization piece next instead.
